# Ringside Analytics — ML Training & Prediction Notebook

## Project Overview
Predicting pro wrestling match outcomes using 40+ years of historical booking data (1980–present).

**Key insight:** We're not predicting who the "better wrestler" is — we're learning what **bookers tend to do** based on patterns like momentum, card position, alignment, and head-to-head history.

### Data Sources
| Source | Records | Content |
|--------|---------|---------|
| ProFightDB (Kaggle) | 363K matches | Multi-era, multi-promotion match results |
| WWE SQLite (Kaggle) | 88K matches | Historical WWE/WWF gap-filling |
| WWE Champion (Kaggle) | 1K matches | Title match enrichment |
| WWE/AEW Ratings (Kaggle) | 12K matches | CageMatch + WON star ratings |
| SmackDown Hotel (scrape) | ~500 records | Face/heel alignment snapshots |
| Smark Out Moment (scrape) | ~600 records | Alignment turn history |

### Feature Set (35 features)
- **Win momentum (5):** 30d/90d/365d win rates, win/loss streaks
- **Event context (4):** PPV flag, title match, card position, event tier
- **Match type (9):** Type-specific win rate + 8 binary flags
- **Title proximity (3):** Champion status, defenses, days since title match
- **Career phase (3):** Years active, recent activity, days since last match
- **Promotion (1):** Promotion-specific win rate
- **Head-to-head (2):** H2H win rate and match count vs opponent
- **Alignment (6):** Face/heel/tweener, days since turn, turn frequency, face-vs-heel matchup
- **Match quality (1):** Running average match rating
- **Card position (1):** Rolling 10-match card position trend

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Resolve MODEL_DIR — three possible sources in priority order:
#   1. Bundled with the notebook (Kaggle ships /kaggle/working/ as the kernel dir)
#   2. Attached Kaggle Model input under /kaggle/input/
#   3. Local development path
def resolve_model_dir() -> Path:
    # Bundled alongside notebook
    for base in (Path("/kaggle/working"), Path.cwd()):
        if (base / "xgboost.joblib").exists():
            return base
    # Attached Kaggle Model
    kinput = Path("/kaggle/input")
    if kinput.exists():
        for p in kinput.glob("**/xgboost.joblib"):
            return p.parent
    # Local dev
    local = Path("/var/www/wrastling/ml/models")
    if (local / "xgboost.joblib").exists():
        return local
    raise FileNotFoundError("No model artifacts found in bundled, input, or local paths.")

MODEL_DIR = resolve_model_dir()
IS_KAGGLE = Path("/kaggle").exists()

print(f"Notebook initialized: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Runtime:         {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Model directory: {MODEL_DIR}")
print(f"Artifacts:       {sorted(f.name for f in MODEL_DIR.glob('*.joblib'))}")


## 1. Dataset Overview

The model was trained on **~197K pro-wrestling matches** spanning WWE, AEW, NXT, WCW, ECW, and TNA (1980–present). The training pipeline runs against the live Ringside Analytics PostgreSQL database; this notebook ships the trained artifacts only, so database tables aren't queried here.

Approximate volumes at the time of training:

| Entity | Count |
|---|---:|
| Wrestlers | 12,800 |
| Events | 27,000 |
| Matches | 201,000 |
| Match participations | 485,000 |
| Title reigns | 5,600 |
| Alignment turns | 631 |

In [ ]:
# Static summary — the actual database lives on the Ringside Analytics server.
# For live data exploration, see https://hookgen.online
print("Promotion breakdown (approx. matches trained on):")
for promo, cnt in [("WWE", 95000), ("WCW", 38000), ("AEW", 22000),
                   ("NXT", 18000), ("ECW", 14000), ("TNA", 14000)]:
    print(f"  {promo:.<15} {cnt:>8,}")


## 2. Training Report — Model Performance

Load the latest training report and display key metrics.

In [ ]:
report_path = MODEL_DIR / "training_report.json"

with open(report_path) as f:
    report = json.load(f)

print("=" * 60)
print("TRAINING REPORT")
print("=" * 60)
print(f"  Trained at:  {report.get('timestamp', 'unknown')}")
print(f"  Best model:  {report.get('best_model', 'unknown')}")
print(f"  Best acc:    {report.get('best_accuracy', 0):.4f}")
print(f"  Best AUC:    {report.get('best_auc', 0):.4f}")

for m in report.get("models", []):
    val = m.get("validation", {})
    test = m.get("test", {})
    print(f"\n  {m['name'].upper()}")
    print(f"    Validation  — acc: {val.get('accuracy', 0):.4f}  AUC: {val.get('auc_roc', 0):.4f}  log-loss: {val.get('log_loss', 0):.4f}")
    print(f"    Test        — acc: {test.get('accuracy', 0):.4f}  AUC: {test.get('auc_roc', 0):.4f}  log-loss: {test.get('log_loss', 0):.4f}")

print("\n  Note: the val→test gap indicates temporal overfitting. Treat")
print("        test metrics as the honest generalization estimate.")


## 3. Model Inspection

Rather than calling a live DB-backed prediction engine, we inspect the trained XGBoost model directly — loading the artifact, reading its feature importances, and generating predictions from synthetic feature vectors.

In [ ]:
import joblib

# FEATURE_COLUMNS is the 35-dim input vector the model expects.
# Kept inline here so the notebook is self-contained on Kaggle.
FEATURE_COLUMNS = [
    "win_rate_30d", "win_rate_90d", "win_rate_365d",
    "current_win_streak", "current_loss_streak",
    "is_ppv", "is_title_match", "card_position", "event_tier",
    "match_type_win_rate",
    "is_singles", "is_tag_team", "is_triple_threat", "is_fatal_four_way",
    "is_ladder", "is_cage", "is_hell_in_a_cell", "is_royal_rumble",
    "is_champion", "num_defenses", "days_since_title_match",
    "years_active", "matches_last_90d", "days_since_last_match",
    "promotion_win_rate",
    "h2h_win_rate", "h2h_matches",
    "alignment", "is_face", "is_heel",
    "days_since_turn", "turns_12m", "face_heel_matchup",
    "avg_match_rating", "card_position_momentum",
]
assert len(FEATURE_COLUMNS) == 35

model = joblib.load(MODEL_DIR / "xgboost.joblib")
print(f"Loaded: {type(model).__name__}")
print(f"Expects {model.n_features_in_} features.")


In [ ]:
def sample_features(persona: str) -> np.ndarray:
    """Hand-crafted feature vectors for illustrative predictions.

    This is NOT how the production API works — the real pipeline computes these
    35 values from the live database at request time. Here we just show how the
    trained model reacts to different archetypes.
    """
    base = dict.fromkeys(FEATURE_COLUMNS, 0.0)
    base.update({"is_singles": 1.0, "event_tier": 1.0, "card_position": 0.5,
                 "years_active": 5.0, "matches_last_90d": 20.0,
                 "days_since_last_match": 7.0, "avg_match_rating": 7.0})

    if persona == "hot_champion":
        base.update(win_rate_30d=0.85, win_rate_90d=0.80, win_rate_365d=0.72,
                    current_win_streak=8, is_champion=1, num_defenses=5,
                    is_title_match=1, is_ppv=1, card_position=1.0,
                    h2h_win_rate=0.7, h2h_matches=3, is_face=1, alignment=1,
                    card_position_momentum=0.3, match_type_win_rate=0.78,
                    promotion_win_rate=0.74)
    elif persona == "cold_challenger":
        base.update(win_rate_30d=0.35, win_rate_90d=0.45, win_rate_365d=0.55,
                    current_loss_streak=3, is_title_match=1, is_ppv=1,
                    card_position=0.9, h2h_win_rate=0.3, h2h_matches=3,
                    is_heel=1, alignment=-1, face_heel_matchup=1,
                    match_type_win_rate=0.52, promotion_win_rate=0.58)
    elif persona == "mid_card":
        base.update(win_rate_30d=0.55, win_rate_90d=0.52, win_rate_365d=0.50,
                    card_position=0.4, match_type_win_rate=0.50,
                    promotion_win_rate=0.51)
    else:
        raise ValueError(persona)

    return np.array([base[c] for c in FEATURE_COLUMNS]).reshape(1, -1)


def predict(p1: str, p2: str):
    x1 = sample_features(p1)
    x2 = sample_features(p2)
    # Production averages the per-side model outputs; here we show both for clarity.
    prob1 = model.predict_proba(x1)[0, 1]
    prob2 = model.predict_proba(x2)[0, 1]
    total = prob1 + prob2
    norm1, norm2 = prob1 / total, prob2 / total
    print(f"\n  {p1:<25} win prob: {norm1:.1%}  {'█' * int(norm1 * 40)}")
    print(f"  {p2:<25} win prob: {norm2:.1%}  {'█' * int(norm2 * 40)}")


In [ ]:
print('Sample matchup: Hot Champion vs Cold Challenger (PPV title match)')
predict('hot_champion', 'cold_challenger')

print('\nSample matchup: Hot Champion vs Mid-Card (regular singles)')
predict('hot_champion', 'mid_card')

print('\nSample matchup: Mid-Card vs Mid-Card (toss-up)')
predict('mid_card', 'mid_card')


### Sample Predictions
Edit these cells to test any matchup you want.

The predictions above use **synthetic feature vectors** rather than live wrestler data. For real predictions against named wrestlers, call the production API at `https://hookgen.online/api/predict`.

## 4. Feature Analysis — What Drives Predictions?

In [ ]:
# Full feature importance ranking from the trained XGBoost model.
FACTOR_LABELS = {
    "win_rate_30d": "Recent win rate (last 30 days)",
    "win_rate_90d": "Win rate over last 3 months",
    "win_rate_365d": "Annual win rate",
    "current_win_streak": "Current winning streak",
    "current_loss_streak": "Current losing streak",
    "is_ppv": "PPV match (higher stakes)",
    "is_title_match": "Championship on the line",
    "card_position": "Card position (main event = higher)",
    "event_tier": "Event importance level",
    "match_type_win_rate": "Win rate in this match type",
    "is_singles": "Singles match",
    "is_tag_team": "Tag team match",
    "is_triple_threat": "Triple threat",
    "is_fatal_four_way": "Fatal four-way",
    "is_ladder": "Ladder match",
    "is_cage": "Cage match",
    "is_hell_in_a_cell": "Hell in a Cell",
    "is_royal_rumble": "Royal Rumble",
    "is_champion": "Currently holds a championship",
    "num_defenses": "Number of title defenses",
    "days_since_title_match": "Days since last title match",
    "years_active": "Years of career experience",
    "matches_last_90d": "Recent activity (matches in 90 days)",
    "days_since_last_match": "Days since last match",
    "promotion_win_rate": "Win rate in this promotion",
    "h2h_win_rate": "Head-to-head record vs opponent",
    "h2h_matches": "Number of prior meetings",
    "alignment": "Current alignment (face/tweener/heel)",
    "is_face": "Wrestling as a face",
    "is_heel": "Wrestling as a heel",
    "days_since_turn": "Days since last alignment turn",
    "turns_12m": "Alignment turns in last 12 months",
    "face_heel_matchup": "Classic face vs heel matchup",
    "avg_match_rating": "Average match quality rating",
    "card_position_momentum": "Card position trend (push trajectory)",
}

fi_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
fi_df["label"] = fi_df["feature"].map(FACTOR_LABELS).fillna(fi_df["feature"])

print("XGBoost Feature Importance — Full Ranking")
print("=" * 70)
for _, row in fi_df.iterrows():
    bar = "█" * int(row["importance"] * 200)
    print(f"  {row['label']:.<48} {row['importance']:.4f} {bar}")


## 5. Training Log — Learnings & Observations

### 2026-03-08 — Initial Training Run (v1)
- **Data:** 525K match participation records across WWE, WCW, ECW, NXT, TNA, AEW
- **Features:** 35 features covering momentum, context, matchups, alignment, quality
- **Split:** Temporal (train < 2024, val = 2024, test = 2025+)
- **Models:** Logistic Regression (baseline) + XGBoost (primary)

**Optimization notes:**
- H2H feature computation was O(n²) — optimized to O(n) with incremental dict accumulator
- Card position momentum vectorized with pandas groupby+rolling
- Title proximity still slow (~9min) due to per-row date lookups — candidate for next optimization

**Known limitations:**
- Alignment data sparse (503 snapshots, 631 turns) — mostly WWE, limited AEW coverage
- No storyline/feud context — model can't know "this is the blowoff match"
- Tag team prediction limited — H2H only computed for 2-person matches
- ProFightDB match ratings skewed negative (some Meltzer ratings < 0, clamped to 0)